# COSC 424/525 – Deep Learning, Spring 2026
## Project 3: Build a GPT from Scratch – From Tiny Shakespeare to Vision Transformers

**Objective:** In this project you will implement a decoder-only Transformer (GPT-style) from scratch and train it to generate Shakespearean text one character at a time. The project has two parts. Part 1 builds a Transformer from scratch for text generation. Part 2 applies a pretrained Vision Transformer to image classification, comparing two transfer learning strategies.

**[424 + 525]** = required for both sections. **[525 ONLY]** = required only for COSC 525 (424 students welcome to try).

### Rules you should note:
- You are NOT allowed to use `torch.nn.Transformer`, `torch.nn.MultiheadAttention`, or any pre-built attention library (e.g. HuggingFace) in Part I.
- This notebook runs on CPU, CUDA GPU, and Apple M-series GPU.
- Do note change the Hyperparameters.
- Your implementation should pass all test cells. Do not modify test cells.


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import math, time, os

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda


---
# Part I: Train a GPT from Scratch – Tiny Shakespeare [424 + 525]

Train a decoder-only Transformer on next-character prediction: given a context window of characters, predict the next character.

Context: `"To be or not to b"` -> Predict: `"e"`

At inference time, sample autoregressively: feed the model a seed string, sample the next character from the output distribution, append it to the context, and repeat. With a well-trained model, the output will exhibit Shakespearean vocabulary, meter, and even character names.

## Hyperparameters (DO NOT MODIFY)

- Context window: 128 characters
- Model dimension d: 128
- Heads h: 4
- FFN inner dim: 256
- Layers N: 3
- Dropout: 0.1
- Warmup steps: 500
- Batch size: 32
- Training steps: 5000
- Optimizer: AdamW (lr=3e-4)

In [7]:
CONTEXT_LEN = 128
D_MODEL     = 128
N_HEADS     = 4
D_FF        = 256
N_LAYERS    = 3
DROPOUT     = 0.1
WARMUP_STEPS = 500
BATCH_SIZE  = 32
TRAIN_STEPS = 5000
LR          = 3e-4
D_K = D_MODEL // N_HEADS
print(f"d_k={D_K}, {N_LAYERS} layers, {N_HEADS} heads")

d_k=32, 3 layers, 4 heads


## Dataset

Download `input.txt` (~1 MB, 40,000 lines) which is the complete works of Shakespeare as plain text. Your model operates at the character level — the vocabulary is the set of unique characters in the file (~65 characters including punctuation, spaces, and newlines). No tokenizer needed.

In [8]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    import urllib.request
    urllib.request.urlretrieve(url, "input.txt")

with open("input.txt") as f:
    text = f.read()

chars = sorted(set(text))
VOCAB_SIZE = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - CONTEXT_LEN, (BATCH_SIZE,))
    x = torch.stack([d[i:i+CONTEXT_LEN] for i in ix])
    y = torch.stack([d[i+1:i+CONTEXT_LEN+1] for i in ix])
    return x.to(device), y.to(device)

print(f"{len(text):,} chars, vocab={VOCAB_SIZE}, train={len(train_data):,}, val={len(val_data):,}")

1,115,394 chars, vocab=65, train=1,003,854, val=111,540


---
## Task 1 — Scaled Dot-Product Attention [424 + 525]

Implement `scaled_dot_product_attention(Q, K, V, mask)` as a standalone function. The causal mask is always required in a decoder-only model — every position can only attend to previous positions.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

In [ ]:
def scaled_dot_product_attention(Q, K : torch.Tensor, V, mask=None):
    """
    Args:
        Q, K, V: (batch, heads, seq_len, d_k)
        mask: bool (1, 1, seq_len, seq_len), True = blocked
    Returns:
        output:  (batch, heads, seq_len, d_k)
        weights: (batch, heads, seq_len, seq_len)
    """
    d_k = Q.size(-1)

    # Step 1: Compute raw attention scores
    #   Hint: matrix multiply Q with transposed K, then scale
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)

    # Step 2: Apply causal mask — set masked positions to -inf
    #   Hint: use .masked_fill(mask, float('-inf'))
    if mask is not None:
        scores.masked_fill_(mask, float('-inf'))

    # Step 3: Softmax over the last dimension to get attention weights
    weights = torch.softmax(scores, dim=-1)

    # Step 4: Weighted sum of the values
    #   Hint: matrix multiply weights with V
    output = torch.matmul(weights, V)

    return output, weights

In [15]:
# ⚠️ DO NOT MODIFY THIS CELL — autograder test cell
# Task 1 Tests
B_t, H_t, T_t, dk_t = 2, 4, 8, 32
Q_t = torch.randn(B_t, H_t, T_t, dk_t)
K_t = torch.randn(B_t, H_t, T_t, dk_t)
V_t = torch.randn(B_t, H_t, T_t, dk_t)
mask_t = torch.triu(torch.ones(T_t, T_t), diagonal=1).bool().unsqueeze(0).unsqueeze(0)

out_t, w_t = scaled_dot_product_attention(Q_t, K_t, V_t, mask=mask_t)

assert out_t.shape == (B_t, H_t, T_t, dk_t), f"Output shape wrong: {out_t.shape}"
assert w_t.shape == (B_t, H_t, T_t, T_t), f"Weights shape wrong: {w_t.shape}"
assert torch.allclose(w_t.sum(dim=-1), torch.ones(B_t, H_t, T_t), atol=1e-4), \
    "Attention weights should sum to 1 along last dim"


print("Task 1 passed")

Task 1 passed


---
## Task 2 — Multi-Head Attention [424 + 525]

Implement a `MultiHeadAttention` module with Q, K, V, and output projection matrices. Unlike the encoder-decoder Transformer from lecture, there is no cross-attention here — only masked self-attention.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # Step 1: Define 4 linear projections (d_model -> d_model)
        #   Hint: nn.Linear(d_model, d_model) for each of W_q, W_k, W_v, W_o
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        assert(C == self.d_model)

        # Step 2: Project inputs and reshape to (B, n_heads, T, d_k)
        #   Hint: project with linear layer, then view as (B, T, n_heads, d_k) and swap dims 1,2
        Q = self.W_q(x).reshape(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).reshape(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).reshape(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Step 3: Call scaled_dot_product_attention from Task 1
        attn_out = scaled_dot_product_attention(Q, K, V, mask)
        attn_weights = attn_out[1]

        # Step 4: Concat heads back to (B, T, d_model) and project
        #   Hint: reverse the reshape from Step 2, then apply the output projection
        output = self.W_o(attn_out[0].transpose(1, 2).reshape(B, T, C))

        return output, attn_weights

In [19]:
# ⚠️ DO NOT MODIFY THIS CELL — autograder test cell
# Task 2 Tests
mha = MultiHeadAttention(D_MODEL, N_HEADS, DROPOUT).to(device)
x_t2 = torch.randn(2, CONTEXT_LEN, D_MODEL, device=device)
cm = torch.triu(torch.ones(CONTEXT_LEN, CONTEXT_LEN, device=device), diagonal=1).bool().unsqueeze(0).unsqueeze(0)

out2, w2 = mha(x_t2, mask=cm)
assert out2.shape == (2, CONTEXT_LEN, D_MODEL), f"Output shape wrong: {out2.shape}"
assert w2.shape == (2, N_HEADS, CONTEXT_LEN, CONTEXT_LEN), f"Weights shape wrong: {w2.shape}"


print(f"MHA params: {sum(p.numel() for p in mha.parameters()):,}")
print("Task 2 passed")
del mha, x_t2, out2, w2

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

---
## Task 3 — Sinusoidal Positional Encoding [424 + 525]

Implement sinusoidal positional encoding. Include a heatmap visualization of the encoding matrix for your context window length.

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Step 1: Compute the div_term in log-space for numerical stability
        #   Hint: compute 1/10000^(2i/d) using exp and log for stability
        div_term = None  # TODO

        # Step 2: Fill even columns (0::2) with sin, odd columns (1::2) with cos
        #   Hint: multiply position by div_term, then apply sin/cos
        pe[:, 0::2] = None  # TODO
        pe[:, 1::2] = None  # TODO

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

        raise NotImplementedError()

    def forward(self, x):
        # Step 3: Add positional encoding to input
        #   Hint: slice self.pe to match the sequence length, then add to x

        raise NotImplementedError()

        return self.dropout(x)

In [ ]:
# ⚠️ DO NOT MODIFY THIS CELL — autograder test cell
# Task 3 Tests
pe_mod = PositionalEncoding(D_MODEL, max_len=CONTEXT_LEN, dropout=0.0)
pe_mat = pe_mod.pe.squeeze(0).numpy()

assert pe_mat.shape == (CONTEXT_LEN, D_MODEL), f"PE shape wrong: {pe_mat.shape}"

x_pe = torch.randn(2, CONTEXT_LEN, D_MODEL)
out_pe = pe_mod(x_pe)
assert out_pe.shape == x_pe.shape, f"Forward shape wrong: {out_pe.shape}"
assert not torch.equal(out_pe, x_pe), "PE not being added to input"


fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(pe_mat, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
ax.set_xlabel("Dimension"); ax.set_ylabel("Position")
ax.set_title("Positional Encoding"); plt.colorbar(ax.images[0], ax=ax)
plt.tight_layout(); plt.show()
print("Task 3 passed")

---
## Task 4 — Decoder Block [424 + 525]

Implement ONE decoder block: masked self-attention → Add & Norm → FFN → Add & Norm. Note: in a decoder-only model there is no cross-attention sublayer.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        # Step 1: Build nn.Sequential: Linear -> ReLU -> Dropout -> Linear -> Dropout
        #   Hint: nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), ...)
        self.net = None  # TODO

        raise NotImplementedError()

    def forward(self, x):
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        # Step 2: Create submodules
        #   Hint: MultiHeadAttention, FeedForward, two nn.LayerNorm
        self.self_attn = None  # TODO
        self.ffn       = None  # TODO
        self.norm1     = None  # TODO
        self.norm2     = None  # TODO
        self.dropout   = nn.Dropout(dropout)

        raise NotImplementedError()

    def forward(self, x, mask=None):
        # Step 3: Sublayer 1 — self-attention + residual + norm
        #   Hint: apply attention, then add residual connection with dropout, then layer norm
        attn_out = None     # TODO
        attn_weights = None # TODO
        x = None            # TODO

        # Step 4: Sublayer 2 — FFN + residual + norm
        #   Hint: same pattern as sublayer 1 but with FFN instead of attention
        ff_out = None  # TODO
        x = None       # TODO

        raise NotImplementedError()

        return x, attn_weights

In [ ]:
# ⚠️ DO NOT MODIFY THIS CELL — autograder test cell
# Task 4 Tests
blk = DecoderBlock(D_MODEL, N_HEADS, D_FF, DROPOUT).to(device)
x_t4 = torch.randn(2, CONTEXT_LEN, D_MODEL, device=device)
cm4 = torch.triu(torch.ones(CONTEXT_LEN, CONTEXT_LEN, device=device), diagonal=1).bool().unsqueeze(0).unsqueeze(0)

out4, w4 = blk(x_t4, mask=cm4)
assert out4.shape == x_t4.shape, f"Output shape wrong: {out4.shape}"
assert w4.shape == (2, N_HEADS, CONTEXT_LEN, CONTEXT_LEN), f"Weights shape wrong: {w4.shape}"


print(f"DecoderBlock params: {sum(p.numel() for p in blk.parameters()):,}")
print("Task 4 passed")
del blk, x_t4, out4, w4

---
## Task 5 — Full GPT Model [424 + 525]

Stack N decoder blocks. Add a character embedding layer at the input and a linear projection to vocabulary size at the output.

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers,
                 max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Step 1: Define layers
        #   Hint: Embedding, PositionalEncoding, ModuleList of DecoderBlocks, LayerNorm, Linear
        self.token_emb = None  # TODO
        self.pos_enc   = None  # TODO
        self.blocks    = None  # TODO
        self.norm      = None  # TODO
        self.head      = None  # TODO

        raise NotImplementedError()

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Step 2: Embed tokens and add positional encoding
        #   Hint: embed the token indices, then add positional information
        x = None  # TODO

        # Step 3: Create causal mask — upper triangular bool matrix
        #   Hint: use torch.triu with diagonal=1 to make a TxT mask, then add batch and head dims
        mask = None  # TODO

        # Step 4: Pass through decoder blocks, collect attn weights
        self._last_attn_weights = []
        # TODO: loop over self.blocks

        # Step 5: Final LayerNorm + linear head -> logits
        logits = None  # TODO

        # Step 6: Compute cross-entropy loss if targets given, else None
        #   Hint: flatten logits and targets before passing to F.cross_entropy
        loss = None

        raise NotImplementedError()

        return logits, loss

In [ ]:
# ⚠️ DO NOT MODIFY THIS CELL — autograder test cell
# Task 5 Tests
model = GPT(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS, CONTEXT_LEN, DROPOUT).to(device)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

xb, yb = get_batch("train")
logits, loss = model(xb, targets=yb)

assert logits.shape == (BATCH_SIZE, CONTEXT_LEN, VOCAB_SIZE), f"Logits shape wrong: {logits.shape}"
assert loss is not None and loss.dim() == 0, "Loss should be a scalar when targets given"
assert loss.item() > 0, "Loss should be positive"


print(f"Init loss: {loss.item():.4f}")
print("Task 5 passed")

---
## Task 6 — Training Loop [424 + 525]

Implement cross-entropy loss over all positions in the context window, AdamW optimizer, and the learning rate warmup + decay schedule.

In [ ]:
def get_lr(step, warmup_steps, max_steps, max_lr):
    """Linear warmup + cosine decay. Returns lr for given step."""
    # Step 1: If in warmup phase, linearly increase from 0 to max_lr
    #   Hint: scale max_lr by the fraction of warmup completed

    # Step 2: Otherwise, cosine decay from max_lr toward 0
    #   Hint: compute decay_ratio, then 0.5 * (1 + cos(pi * ratio))

    raise NotImplementedError()

In [ ]:
# ⚠️ DO NOT MODIFY THIS CELL — autograder test cell
# Task 6 LR Tests
lr_0 = get_lr(0, 500, 5000, 3e-4)
lr_500 = get_lr(500, 500, 5000, 3e-4)
lr_mid = get_lr(2500, 500, 5000, 3e-4)

assert isinstance(lr_0, float), "get_lr should return a float"
assert lr_500 > lr_0, "LR should increase during warmup"
assert lr_mid < lr_500, "LR should decay after warmup"
assert lr_mid > 0, "LR should be positive during training"


lrs = [get_lr(s, WARMUP_STEPS, TRAIN_STEPS, LR) for s in range(TRAIN_STEPS)]
plt.figure(figsize=(10, 3)); plt.plot(lrs); plt.xlabel("Step"); plt.ylabel("LR")
plt.title("LR Schedule"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("Task 6 LR passed")

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=50):
    model.eval()
    losses = {}
    for split in ["train", "val"]:
        batch_losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, targets=yb)
            batch_losses.append(loss.item())
        losses[split] = np.mean(batch_losses)
    model.train()
    return losses

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
LOG_INTERVAL  = 100
EVAL_INTERVAL = 500

train_losses, val_losses, step_log = [], [], []

model.train()
start_time = time.time()

for step in range(TRAIN_STEPS):
    lr = get_lr(step, WARMUP_STEPS, TRAIN_STEPS, LR)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    xb, yb = get_batch("train")

    # Step 1: Forward pass — get logits and loss from model
    logits, loss = None, None  # TODO

    # Step 2: Backward pass — zero grads, then loss.backward()
    # TODO

    # Step 3: Optimizer step
    # TODO

    raise NotImplementedError()

    if step % LOG_INTERVAL == 0:
        train_losses.append(loss.item())
        print(f"Step {step:5d} | loss {loss.item():.4f} | lr {lr:.2e} | {time.time()-start_time:.0f}s")
    if step % EVAL_INTERVAL == 0 or step == TRAIN_STEPS - 1:
        ev = estimate_loss(model)
        val_losses.append(ev["val"]); step_log.append(step)
        print(f"  eval | train {ev['train']:.4f} | val {ev['val']:.4f}")
    if step == 500:
        checkpoint_500 = {k: v.clone() for k, v in model.state_dict().items()}

print(f"\nDone in {(time.time()-start_time)/60:.1f} min")

**Training Curves**

A plot of training and validation loss vs. steps is shown below. Compare a sample generated after 500 steps vs. 5,000 steps. What has the model learned to imitate — character names, scene headers, dialogue format?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(0, len(train_losses)*LOG_INTERVAL, LOG_INTERVAL), train_losses, label="Train", alpha=0.7)
ax.plot(step_log, val_losses, label="Val", marker="o")
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("Loss vs Steps")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Task 7 — Autoregressive Sampling [424 + 525]

Implement autoregressive sampling with a temperature parameter. Temperature < 1 makes output more conservative; temperature > 1 makes it more creative (and more chaotic). Show generated samples at temperature 0.8 and 1.2.

In [ ]:
@torch.no_grad()
def generate(model, seed_text, max_new_chars=200, temperature=1.0):
    model.eval()
    idx = torch.tensor(encode(seed_text), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_chars):
        idx_cond = idx[:, -CONTEXT_LEN:]

        # Step 1: Get logits from model
        #   Hint: model(idx_cond) returns (logits, loss)
        logits = None  # TODO

        # Step 2: Take logits at last position, divide by temperature
        logits_last = None  # TODO

        # Step 3: Softmax to get probabilities
        probs = None  # TODO

        # Step 4: Sample next character index
        #   Hint: torch.multinomial(probs, num_samples=1)
        next_idx = None  # TODO

        raise NotImplementedError()

        idx = torch.cat([idx, next_idx], dim=1)

    model.train()
    return decode(idx[0].tolist())

**Generated Samples**

At least 3 generated passages of 200+ characters each, produced at different temperatures. Show samples at temperature 0.8 and 1.2. Write your observations below.

In [ ]:
seed = "\nTo be or not to be"
for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"\n{'='*60}\n  Temperature = {temp}\n{'='*60}")
    print(generate(model, seed, max_new_chars=300, temperature=temp))

---
### 500 Steps vs. 5,000 Steps

Compare a sample generated after 500 steps vs. 5,000 steps. What has the model learned to imitate — character names, scene headers, dialogue format?

In [ ]:
# Load the 500-step checkpoint, generate, then restore the final model
final_state = {k: v.clone() for k, v in model.state_dict().items()}

model.load_state_dict(checkpoint_500)
print("=" * 60)
print("  Sample after 500 steps")
print("=" * 60)
print(generate(model, seed, max_new_chars=300, temperature=0.8))

model.load_state_dict(final_state)
print("\n" + "=" * 60)
print(f"  Sample after {TRAIN_STEPS} steps")
print("=" * 60)
print(generate(model, seed, max_new_chars=300, temperature=0.8))

---
### What happens without the causal mask?

What happens to output quality if you remove the causal mask entirely? Why?

In [ ]:
# Temporarily remove the causal mask and generate text.
# Compare the output with and without the mask.

# TODO: your implementation here

raise NotImplementedError()

**Your analysis (causal mask removal):**

Write 1-2 paragraphs here.

---
---
# 525 ONLY — Tasks 8 and 9

Not required for 424. 424 students are welcome to try.

## Task 8 — Attention Weight Visualization [525 ONLY]

Extract and visualize the attention weight matrices from each head across several layers for a fixed input passage. Do different heads attend to different character-level patterns? Do any heads appear to track punctuation, spaces, or repeated words? Write a 1–2 paragraph analysis.

In [ ]:
@torch.no_grad()
def get_attention_maps(model, input_text):
    """Return list of N_LAYERS tensors, each (n_heads, seq, seq)."""
    model.eval()
    idx = torch.tensor(encode(input_text), dtype=torch.long, device=device).unsqueeze(0)

    # Step 1: Forward pass to populate model._last_attn_weights
    # Step 2: Return the weights, squeeze out the batch dim
    #   Hint: run a forward pass, then collect weights from model._last_attn_weights

    raise NotImplementedError()

In [ ]:
passage = text[:CONTEXT_LEN]
all_attn = get_attention_maps(model, passage)
CROP = 50
fig, axes = plt.subplots(N_LAYERS, N_HEADS, figsize=(4*N_HEADS, 4*N_LAYERS))
if N_LAYERS == 1: axes = [axes]
for li in range(N_LAYERS):
    for hi in range(N_HEADS):
        ax = axes[li][hi] if N_HEADS > 1 else axes[li]
        ax.imshow(all_attn[li][hi, :CROP, :CROP].cpu().numpy(), cmap="viridis", aspect="auto")
        ax.set_title(f"L{li+1} H{hi+1}", fontsize=9)
plt.suptitle("Attention Heatmaps", y=1.02); plt.tight_layout(); plt.show()

**Your analysis (Task 8):**

Do different heads attend to different character-level patterns? Do any heads appear to track punctuation, spaces, or repeated words?

---
## Task 9 — Sampling Strategy Comparison [525 ONLY]

Compare three sampling strategies: greedy decoding (always pick the most likely next character), temperature sampling, and top-k sampling. Generate 10 passages with each method and evaluate coherence qualitatively. At what temperature does output transition from repetitive to creative to incoherent?

In [ ]:
@torch.no_grad()
def generate_greedy(model, seed_text, max_new_chars=200):
    model.eval()
    idx = torch.tensor(encode(seed_text), dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_chars):
        idx_cond = idx[:, -CONTEXT_LEN:]
        # Get logits, pick argmax of last position
        #   Hint: take the last position's logits and select the token with highest probability

        raise NotImplementedError()

        idx = torch.cat([idx, next_idx], dim=1)
    return decode(idx[0].tolist())


@torch.no_grad()
def generate_topk(model, seed_text, max_new_chars=200, k=10, temperature=1.0):
    model.eval()
    idx = torch.tensor(encode(seed_text), dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_chars):
        idx_cond = idx[:, -CONTEXT_LEN:]
        # Get logits, apply temperature, keep top-k, set rest to -inf, sample
        #   Hint: torch.topk to get top values, mask out below threshold, softmax, multinomial

        raise NotImplementedError()

        idx = torch.cat([idx, next_idx], dim=1)
    return decode(idx[0].tolist())

In [ ]:
seed = "\nTo be or not to be"
for name, fn in [("GREEDY", lambda: generate_greedy(model, seed)),
                  ("TEMP=0.8", lambda: generate(model, seed, temperature=0.8)),
                  ("TOP-K=10", lambda: generate_topk(model, seed, k=10))]:
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    for i in range(3):
        print(f"--- sample {i+1} ---"); print(fn())

**Your analysis (Task 9):**

Compare the three strategies. At what temperature does output transition from repetitive to creative to incoherent?

---
---
# Part II: ViT – From Feature Extraction to Fine-Tuning [424 + 525]

In Part I, you built a Transformer from scratch. In Part II, you will use a pretrained Vision Transformer (ViT) from HuggingFace and apply it to image classification — without training from scratch. You will compare two transfer learning strategies on the same dataset and analyze what the model has learned visually.

You may use HuggingFace transformers for this part.

**Dataset:** Oxford 102 Flowers. 102 flower species, ~8,000 images at variable resolution. Loads via `datasets.load_dataset("nelorth/oxford-flowers")`. This dataset is more challenging than CIFAR-10, but better for seeing the value of pretrained features.

**Model:** Use ViT-Base model pretrained on ImageNet-21k, with 16×16 patch size and d = 768. You will need to replace the classification head with one matching the number of classes of Oxford 102 Flowers.

Runs on CPU, CUDA, and Apple M-series GPU. Batch size auto-adjusts on CPU.

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor
from datasets import load_dataset
from torchvision import transforms

VIT_MODEL_NAME  = "google/vit-base-patch16-224-in21k"
NUM_CLASSES     = 102
VIT_EPOCHS      = 5
VIT_BATCH_SIZE  = 32
VIT_LR_HEAD     = 1e-3
VIT_LR_FINETUNE = 2e-5
WARMUP_FRACTION = 0.10
IMAGE_SIZE      = 224

if device.type == "cpu":
    VIT_BATCH_SIZE = 16
    print(f"CPU detected — batch size = {VIT_BATCH_SIZE}")

print(f"Device: {device}")

---
## Data Preparation [424 + 525]

Load the Oxford 102 Flowers dataset, create train / val / test splits, build image transforms, implement a custom Dataset class, and create DataLoaders.

Notes:
- The HuggingFace dataset `nelorth/oxford-flowers` has only `train` and `test` splits — you need to split `train` into train / val yourself
- Use `ViTImageProcessor` to obtain the correct normalization mean and std
- Set `num_workers=0` on CPU / MPS to avoid multiprocessing issues

**Deliverable:** `train_loader`, `val_loader`, `test_loader` ready for training.

In [ ]:
# Load the Oxford Flowers dataset, create train/val/test splits,
# build transforms, a custom Dataset class, and DataLoaders.
#
# Your code must define: train_loader, val_loader, test_loader

# TODO: your implementation here

raise NotImplementedError()

In [ ]:
# ⚠️ DO NOT MODIFY THIS CELL — autograder test cell
# Data preparation tests — do not modify
assert 'train_loader' in dir() and train_loader is not None
assert 'val_loader'   in dir() and val_loader   is not None
assert 'test_loader'  in dir() and test_loader  is not None
_imgs, _lbls = next(iter(train_loader))
assert _imgs.ndim == 4, f"Expected 4D tensor, got {_imgs.ndim}D"
assert _imgs.shape[1] == 3, f"Expected 3 channels, got {_imgs.shape[1]}"
assert _imgs.shape[2] == IMAGE_SIZE and _imgs.shape[3] == IMAGE_SIZE


print(f"Data check passed — batch shape: {_imgs.shape}")

---
## Task 10 — Feature Extraction (Frozen Backbone) [424 + 525]

Freeze all ViT weights. Replace only the final classification head with a new linear layer matching your number of classes. Train only the head for 5 epochs. The ViT acts as a fixed feature extractor — no attention weights change.

**Deliverables:**
- `vit_frozen` model with frozen backbone and trainable head
- Print total vs trainable parameter counts

In [ ]:
# Build a feature extractor: load pretrained ViT, freeze backbone, replace head.
# Remember to update model.config.num_labels after replacing the head.
#
# Must define: vit_frozen, total_p, train_p

# TODO: your implementation here

raise NotImplementedError()

In [ ]:
# ⚠️ DO NOT MODIFY THIS CELL — autograder test cell
# Task 10 model tests — do not modify
assert train_p < total_p, "Backbone should be frozen"
assert train_p > 0, "Head should have trainable params"


print("Task 10 model check passed")

### Training & Evaluation Functions

Write a training function and an evaluation function.

**Signatures:**
- `train_one_epoch_vit(model, loader, optimizer, scheduler=None)` -> `(avg_loss, accuracy, elapsed_seconds)`
- `evaluate_vit(model, loader)` -> `(avg_loss, accuracy)`

The ViT model accepts `model(pixel_values=images, labels=labels)` and returns an object with `.loss` and `.logits`.

In [ ]:
# Implement train_one_epoch_vit and evaluate_vit with the signatures above.

# TODO: your implementation here

raise NotImplementedError()

### Feature-Extraction Training Loop

Train `vit_frozen` for `VIT_EPOCHS` epochs. Use AdamW on the trainable parameters only (`VIT_LR_HEAD`).

**Deliverable:**
- `history_frozen` dict with keys `train_loss`, `val_loss`, `train_acc`, `val_acc`, `epoch_time`
- Print epoch-wise metrics

In [ ]:
# Train vit_frozen for VIT_EPOCHS epochs.
# Must define: history_frozen (dict with the keys listed above)

# TODO: your implementation here

raise NotImplementedError()

---
# 525 ONLY — Task 11: Full Fine-Tuning

Unfreeze all weights. Train the entire model (backbone + head) for 5 epochs with a small learning rate (≤ 2e-5). Use a learning rate warmup for the first 10% of steps.

**Deliverables:**
- `vit_finetune` model with all parameters trainable
- `history_finetune` dict (same keys as Task 10)
- Print epoch-wise metrics

In [ ]:
# Build a fine-tune model: load pretrained ViT, replace head, do NOT freeze.
# Must define: vit_finetune, ft_total, ft_train

# TODO: your implementation here

raise NotImplementedError()

In [ ]:
from torch.optim.lr_scheduler import LambdaLR

# Fine-tune vit_finetune for VIT_EPOCHS with AdamW + linear warmup.
# Must define: history_finetune (dict with same keys as Task 10)

# TODO: your implementation here

raise NotImplementedError()

---
## Comparison [424 + 525]

Based on Tasks 10 (both sections) and 11 (525 only), record and plot:
1. Training and validation loss per epoch
2. Training and validation accuracy per epoch
3. Training time per epoch
4. Total trainable parameters

Write your findings below.

In [ ]:
# Create comparison plots using history_frozen (and history_finetune for 525).
# Include: loss curves, accuracy curves, time per epoch, parameter counts.

# TODO: your implementation here

raise NotImplementedError()

**Your findings (Tasks 10 and 11):**

Write your comparison findings here. 424: discuss Task 10 results. 525: compare both tasks.

---
## ViT Attention Visualization [424 + 525]

For at least 5 correctly classified test images, extract and visualize the attention weights from the last transformer layer. Use the [CLS] token's attention to all patch tokens, reshaped back onto the original image as a heatmap overlay. This shows which image regions the model "looked at" to make its prediction.

**Deliverable:** A figure showing original images and their CLS attention heatmaps.

In [ ]:
# Implement the full attention visualization pipeline:
# 1. Load a copy of vit_frozen with attn_implementation="eager"
# 2. Find 5+ correctly classified test images
# 3. For each image, extract CLS attention from the last layer,
#    reshape to a spatial grid, and overlay on the original image
# 4. Display the results

# TODO: your implementation here

raise NotImplementedError()

**Your analysis (ViT transfer learning and attention):**

Findings from Tasks 10 and 11. What does the model attend to in the attention visualizations?